In [9]:
import pandas as pd
from openai import OpenAI
import os
from src.config import QDRANT_URL, QDRANT_API_KEY, OPENAI_API_KEY, OPENAI_MODEL, OPENAI_API_URL,\
     DENSE_EMBEDDING_MODEL_PATH, OPENAI_MODEL_MINI, INTERIM_DATA_DIR, PROCESSED_DATA_DIR, DEVICE, \
     DEEPSEEK_MODEL
from src.dataset import (
    rewrite_query_descriptions_csv,
    get_sqlite_database_path,
    get_schema_description_path,
    parse_schema_description_path,
)
from src.script_generator import (
    generate_query_descriptions,
    generate_related_query_descriptions_csv,
    generate_sql_scripts_and_results,
)
from src.vanna_connector import initialize_vanna

In [10]:
url = os.path.join(PROCESSED_DATA_DIR, "sakila", "sakila_inline_short.sqlite.db")
DATABASE_NAME = "sakila"


sqlite_config = {
    "params": {
        "url": str(url) 
    },
    "type": "sqlite"}

qdrant_config = {"fastembed_model": DENSE_EMBEDDING_MODEL_PATH,
                 "url": QDRANT_URL, 
                 "api_key": QDRANT_API_KEY,
                 "device": DEVICE}

openai_config = {"api_key": OPENAI_API_KEY,
                 "model": OPENAI_MODEL,
                 "base_url": OPENAI_API_URL}

In [11]:
vanna_client = initialize_vanna(db_config=sqlite_config,
                                qdrant_config=qdrant_config,
                                openai_config=openai_config)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3172.92it/s]
/home/user/cursor_projects/vanna-sql/venv/lib/python3.10/site-packages/vanna/legacy/qdrant/qdrant.py:49: UserWarning: Api key is used with an insecure connection.
  self._client = QdrantClient(


In [12]:
openai_client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_URL,
)

In [13]:
schema_description_path = get_schema_description_path(
    database_name=DATABASE_NAME,
    comment_style="inline",
    comment_variant="short",
)

descriptions_csv_path = generate_query_descriptions(
    client=openai_client,
    model=OPENAI_MODEL,
    schema_description_path=schema_description_path,
    counts_by_difficulty={"easy": 40, "medium": 40, "hard": 40},
    temperature=1.0,
)

parsed = parse_schema_description_path(schema_description_path)
sqlite_db_path = get_sqlite_database_path(
    database_name=parsed["database_name"],
    comment_style=parsed["comment_style"],
    comment_variant=parsed["comment_variant"],
)

print("Descriptions CSV:", descriptions_csv_path)
print("Schema description:", schema_description_path)
print("SQLite DB:", sqlite_db_path)

Generating query descriptions: 100%|██████████| 120/120 [15:01<00:00,  7.51s/it]

Descriptions CSV: /home/user/cursor_projects/vanna-sql/data/interim/sakila/query_descriptions/query_descriptions_inline_short.csv
Schema description: /home/user/cursor_projects/vanna-sql/data/interim/sakila/schema_descriptions/schema_description_inline_short.txt
SQLite DB: /home/user/cursor_projects/vanna-sql/data/processed/sakila/sakila_inline_short.sqlite.db


In [14]:
generation_summary = generate_sql_scripts_and_results(
    client=openai_client,
    model=OPENAI_MODEL,
    sqlite_db_path=sqlite_db_path,
    interim_dir=os.path.join(INTERIM_DATA_DIR, DATABASE_NAME),
    descriptions_csv_path=descriptions_csv_path,
    schema_description_path=schema_description_path,
    temperature=0.01,
)

generation_summary

Generating SQL scripts and results: 100%|██████████| 120/120 [51:36<00:00, 25.80s/it]


{'total': 120,
 'generated': 120,
 'saved': 101,
 'failed': 12,
 'empty': 7,
 'invalid_sql': 0}

In [15]:
# descriptions_csv_path = os.path.join(INTERIM_DATA_DIR, "bank_transaction_monitoring", "query_descriptions", "query_descriptions_inline_short.csv")

COMMENT_STYLE = "inline"

rewrite_style_configs = {
    "short": {
        "schema_description_path": get_schema_description_path(
            DATABASE_NAME, COMMENT_STYLE, "short"
        ),
    },
    "business": {
        "schema_description_path": get_schema_description_path(
            DATABASE_NAME, COMMENT_STYLE, "business"
        ),
    },
    "technical": {
        "schema_description_path": get_schema_description_path(
            DATABASE_NAME, COMMENT_STYLE, "technical"
        ),
    },
}


rewritten_descriptions_csv_path = rewrite_query_descriptions_csv(
    descriptions_csv_path=descriptions_csv_path,
    client=openai_client,
    model=OPENAI_MODEL_MINI,
    style_configs=rewrite_style_configs,
    source_column="query",
    temperature=0.9,
)

print("Rewritten descriptions CSV:", rewritten_descriptions_csv_path)


Rewriting query descriptions: 100%|██████████| 360/360 [08:32<00:00,  1.42s/it]

Rewritten descriptions CSV: /home/user/cursor_projects/vanna-sql/data/interim/sakila/query_descriptions/query_descriptions_inline_short_rewritten.csv


In [16]:
# rewritten_descriptions_csv_path = os.path.join(INTERIM_DATA_DIR, "bank_transaction_monitoring", "query_descriptions", "query_descriptions_inline_short_rewritten.csv")

related_style_configs = {
    style: {
        "schema_description_path": cfg["schema_description_path"],
        "source_column": f"query_{style}",
    }
    for style, cfg in rewrite_style_configs.items()
}

related_descriptions_csv_path = generate_related_query_descriptions_csv(
    rewritten_descriptions_csv_path=rewritten_descriptions_csv_path,
    client=openai_client,
    model=DEEPSEEK_MODEL,
    style_configs=related_style_configs,
    temperature=0.9,
    num_queries=3,
)

print("Related descriptions CSV:", related_descriptions_csv_path)


Generating related query descriptions:   0%|          | 0/1080 [00:00<?, ?it/s]

Generating related query descriptions: 100%|██████████| 1080/1080 [4:09:12<00:00, 13.84s/it]   

Related descriptions CSV: /home/user/cursor_projects/vanna-sql/data/interim/sakila/query_descriptions/query_descriptions_inline_short_rewritten_related.csv
